# 04 — Validação temporal e seleção (seções 13–16 do v5)

**Lê:** `painel` (só linhas de treino), `panel_base` até `TRAIN_TARGET_END`, `painel_spec.json`.  
**Grava:** `cv_detalhe`, `resumo_validacao`, **`selecao_modelo.json`** (com o hash do painel).

Nenhuma linha de teste é carregada. A seleção é congelada aqui e destrava o notebook 05.

In [1]:
# === SETUP (idêntico em todos os notebooks; só muda NOTEBOOK) ===
NOTEBOOK = "04_validacao_selecao"
RAIZ = "/content/drive/MyDrive/tcc_fluxo"      # Colab. Localmente: caminho da pasta do projeto (ou deixe None)

import os, sys
try:
    from google.colab import drive
    drive.mount("/content/drive")
    !pip install -q -r {RAIZ}/requirements.txt
except ImportError:                              # fora do Colab
    RAIZ = os.environ.get("TCC_RAIZ") or None
    if RAIZ is None:
        from pathlib import Path
        RAIZ = str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").is_file()))
sys.path.insert(0, f"{RAIZ}/src")
%load_ext autoreload
%autoreload 2

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tcc_fluxo import iniciar
C, A = iniciar(RAIZ, notebook=NOTEBOOK)

AVISO de reprodutibilidade: pandas=3.1.1; referência recomendada=2.2.3
AVISO de reprodutibilidade: numpy=2.5.4; referência recomendada=2.1.3
AVISO de reprodutibilidade: scikit-learn=1.9.1; referência recomendada=1.6.1
Notebook: 04_validacao_selecao | execução: v6_modular | raiz: C:\Users\andre\OneDrive\Documentos\Projetos\Tccv2\tcc_fluxo_modular_v6


In [2]:
from tcc_fluxo import cv, modelos, Especificacao

spec = Especificacao.de_json(A.carregar_json("painel_spec"), C)
train = A.carregar_treino()
hist = A.carregar_historico_treino()
BOOST, RIDGE_NAMES, CV_MODELS = modelos.lista_modelos(C.RIDGE_ALPHAS)
cv_origins, dropped = cv.origens_cv(C)
print("Modelos na validação:", CV_MODELS)
print(f"Origens completas ({len(cv_origins)}): {[o.date() for o in cv_origins]} | removidas: {dropped or 'nenhuma'}")

Modelos na validação: ['Naive', 'MediaMovel4', 'Ridge_a10', 'Ridge_a100', 'Ridge_a1000', 'Ridge_a10000', 'HistGB_MAE', 'HistGB_Poisson', 'LightGBM_MAE', 'LightGBM_MAE_Exog', 'CatBoost_MAE', 'CatBoost_MAE_raw', 'CatBoost_Tweedie', 'CatBoost_MAE_Exog']
Origens completas (6): [datetime.date(2017, 6, 26), datetime.date(2017, 7, 24), datetime.date(2017, 8, 21), datetime.date(2017, 9, 18), datetime.date(2017, 10, 16), datetime.date(2017, 11, 13)] | removidas: [datetime.date(2017, 12, 11)]


## 14. Validação temporal (somente dentro do treino)

In [3]:
cv_detail, ridge_q = cv.rodar_cv(train, hist, cv_origins, CV_MODELS, spec, C)
A.salvar_df(cv_detail, "cv_detalhe")
A.tabela(cv_detail, "validacao_temporal_detalhada.csv")

origem 2017-06-26 | treino  5,543 | val 2,311 (4 sem.) |   152s
origem 2017-07-24 | treino  7,854 | val 2,496 (4 sem.) |   285s
origem 2017-08-21 | treino 10,350 | val 2,706 (4 sem.) |   438s
origem 2017-09-18 | treino 13,056 | val 3,001 (4 sem.) |   591s
origem 2017-10-16 | treino 16,057 | val 3,234 (4 sem.) |   752s
origem 2017-11-13 | treino 19,291 | val 3,416 (4 sem.) |   912s


WindowsPath('C:/Users/andre/OneDrive/Documentos/Projetos/Tccv2/tcc_fluxo_modular_v6/execucoes/v6_modular/saidas/validacao_temporal_detalhada.csv')

## 15. Estabilidade do Ridge
Instável se, em alguma origem, gerar previsão não finita ou acima do maior alvo do treino da própria origem.

In [4]:
display(ridge_q)
ridge_stab, RIDGE_NAME, RIDGE_ELIGIBLE = cv.estabilidade_ridge(cv_detail, RIDGE_NAMES)
display(ridge_stab)
print(f"Ridge {'estável: ' + RIDGE_NAME if RIDGE_ELIGIBLE else 'REMOVIDO da seleção (documentado: ' + RIDGE_NAME + ')'}")

,origem,modelo,q0,q0.5,q0.9,q0.99,q0.999,q1,max_alvo_treino,n_nao_finitas,n_negativas_brutas,pred_max
0,2017-06-26,Ridge_a10,0.0000,2.2530,49.7551,"2,430.9151","196,256.2332","639,204.2074","10,177.6400",0,301,"639,204.2074"
1,2017-06-26,Ridge_a100,0.0000,2.7432,44.9333,"4,157.8025","384,391.3287","1,418,912.7506","10,177.6400",0,99,"1,418,912.7506"
2,2017-06-26,Ridge_a1000,0.5936,4.3953,49.7814,"3,680.1141","506,668.0218","1,888,200.9108","10,177.6400",0,0,"1,888,200.9108"
3,2017-06-26,Ridge_a10000,2.6015,5.8081,26.9296,666.9737,"44,171.7889","99,426.0845","10,177.6400",0,0,"99,426.0845"
4,2017-07-24,Ridge_a10,0.0000,4.4017,90.4670,"2,610.9657","62,557.6434","82,682.2145","10,177.6400",0,8,"82,682.2145"
5,2017-07-24,Ridge_a100,0.3510,5.1402,73.7640,"2,584.5027","132,256.9710","188,300.7954","10,177.6400",0,0,"188,300.7954"
6,2017-07-24,Ridge_a1000,0.9378,6.1275,65.1693,"3,281.4678","126,531.3997","297,657.2629","10,177.6400",0,0,"297,657.2629"
7,2017-07-24,Ridge_a10000,2.6912,6.6866,34.4658,"1,259.8413","70,401.4292","106,549.8681","10,177.6400",0,0,"106,549.8681"
8,2017-08-21,Ridge_a10,0.1017,4.9945,98.8810,"2,351.7911","17,797.4786","126,514.2883","10,177.6400",0,0,"126,514.2883"
9,2017-08-21,Ridge_a100,0.5073,5.6970,75.5305,"2,649.0787","32,693.9208","279,900.7439","10,177.6400",0,0,"279,900.7439"


,MASE_vend_media,WAPE_vend_media,skill_medio,pred_max,max_alvo_treino,origens_acima_max,nao_finitas,estavel
modelo,,,,,,,,
Ridge_a10,"26,169,588.7023","5,073,965.9971","-100,167,228.7546","470,422,051,960,354.5000","16,372.2300",6,0,False
Ridge_a100,"105,883,546.8844","19,608,197.0311","-383,373,271.0322","1,911,939,964,144,805.7500","16,372.2300",6,0,False
Ridge_a1000,"23,399,243.8195","4,337,915.8080","-84,833,377.9424","422,465,052,598,657.1250","16,372.2300",6,0,False
Ridge_a10000,"62,065.0618","81,852.9074","-1,897,788.0691","4,195,937,526,844.3657","16,372.2300",6,0,False


Ridge REMOVIDO da seleção (documentado: Ridge_a10000)


## 16. Resumo da validação e seleção congelada

In [5]:
resumo = cv.resumo_validacao(cv_detail, BOOST, RIDGE_NAME, RIDGE_ELIGIBLE)
A.salvar_df(resumo, "resumo_validacao")
A.tabela(resumo, "resumo_validacao.csv")
display(resumo.sort_values("MASE_vend_media"))

selection = cv.selecionar(resumo, cv_origins, C, RIDGE_NAME, RIDGE_ELIGIBLE, painel_sha256=A.sha("painel"))
selection["boost_models"] = BOOST
A.salvar_json(selection, "selecao_modelo")
print({k: selection[k] for k in ["modelo_escolhido", "criterio", "valor_criterio", "modelos_em_empate_pratico"]})

,modelo,MAE_media,MAE_mediana,MAE_dp,RMSE_media,WAPE_media,WAPE_dp,sMAPE_media,MASE_vend_media,MASE_vend_media_dp,MASE_vend_mediana,WAPE_vend_media,skill_media,skill_dp,prop_vend_melhora,n_origens,familia,candidato
13,CatBoost_MAE_Exog,143.4004,132.5143,32.8255,404.5153,0.7034,0.0214,146.5534,0.9986,0.1871,0.5582,0.9539,0.1877,0.0264,0.7390,6,CatBoost_MAE_Exog,True
10,CatBoost_MAE,144.5374,133.6278,33.4564,410.8409,0.7088,0.0229,143.5322,1.0005,0.1877,0.5583,0.9526,0.1815,0.0298,0.7335,6,CatBoost_MAE,True
9,LightGBM_MAE_Exog,144.5680,134.6203,32.1522,401.1465,0.7099,0.0207,105.8853,1.0023,0.1823,0.5597,0.9765,0.1802,0.0245,0.7096,6,LightGBM_MAE_Exog,True
8,LightGBM_MAE,144.7201,137.1006,31.8569,398.2132,0.7110,0.0280,106.0987,1.0086,0.1784,0.5691,0.9854,0.1793,0.0218,0.7039,6,LightGBM_MAE,True
11,CatBoost_MAE_raw,142.7160,131.0123,33.1098,387.7350,0.6998,0.0201,137.7761,1.0146,0.1820,0.5711,0.9964,0.1917,0.0320,0.7445,6,CatBoost_MAE_raw,True
6,HistGB_MAE,198.1384,182.5472,42.1677,583.7916,0.9745,0.0194,84.4601,1.1440,0.2223,0.5978,1.0034,-0.1258,0.0407,0.6153,6,HistGB_MAE,True
1,MediaMovel4,160.5546,148.8066,32.4054,383.7514,0.7907,0.0158,100.3785,1.1839,0.1855,0.7535,1.2854,0.0869,0.0175,0.6159,6,MediaMovel4,True
12,CatBoost_Tweedie,158.1657,147.3095,29.2238,387.1765,0.7809,0.0249,146.0413,1.2565,0.1377,0.7816,1.2817,0.0983,0.0256,0.6548,6,CatBoost_Tweedie,True
0,Naive,175.7969,166.0043,34.6754,437.9765,0.8664,0.0300,75.3007,1.3261,0.2296,0.7595,1.3692,0.0000,0.0000,0.0000,6,Naive,True
7,HistGB_Poisson,165.6825,157.0566,27.7306,391.4709,0.8201,0.0407,145.7496,1.3899,0.2688,0.8519,1.3780,0.0533,0.0416,0.6151,6,HistGB_Poisson,True


{'modelo_escolhido': 'LightGBM_MAE_Exog', 'criterio': 'menor MASE_vend_media (média entre 6 origens), empate prático <= 1% -> mais simples', 'valor_criterio': 1.0023453177935397, 'modelos_em_empate_pratico': ['LightGBM_MAE_Exog', 'CatBoost_MAE', 'CatBoost_MAE_Exog']}


In [6]:
A.finalizar()

04_validacao_selecao: leu 3 artefato(s), gravou 3.
